## Generación de máscaras esféricas/elipsoidales a partir de coordenadas rCMBs

In [1]:
import os
import nibabel as nib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

# --- CONFIGURACIÓN DE RUTAS ---
BASE_PATH = r"C:\Users\\marta\Downloads\workdir"
IMAGE_DIR = os.path.join(BASE_PATH, "raw/positives")
MASK_DIR = os.path.join(BASE_PATH, "labels/positives")
CSV_PATH = r"C:\Users\marta\Downloads\ADNI_MCH_Clean_Dataset.csv"
OUTPUT_VIZ_DIR = os.path.join(BASE_PATH, "visualizations")

Rehacemos intentando mejorar la máscara para que sea más volumétrica. La dificultad radica en que el dataset tiene baja resolución en el eje z (4mm).

### Resumen de Lógica: v10

Esta versión abandona la suposición de que "todo es una esfera" y se centra en la consistencia radiométrica y espacial entre cortes.
1. Independencia Axial ($X, Y$): Se calculan radios $r_x$ y $r_y$ por separado analizando el perfil de intensidad hasta recuperar el 70% del valor del fondo local. Esto permite capturar morfologías alargadas (sangrados perivasculares).
2. Test de Conectividad Espacial ($Z$): Antes de expandir la máscara a un corte adyacente, el algoritmo busca el punto más oscuro en dicho corte. Solo se considera "misma lesión" si:
    - La distancia radial (en el plano $X, Y$) entre el centro original y el nuevo punto es < 1.0 mm.
    - El contraste sigue siendo significativo ($< 70\%$ del fondo).
3. Anclaje Volumétrico: Si la lesión es persistente en varios cortes, el centroide $Z$ de la elipse se sitúa en el punto medio físico de esos cortes. Esto garantiza que la máscara sea simétrica y que el corte central sea el de mayor diámetro.
4. Compensación de Anisotropía: El radio $r_z$ no es una medida de gradiente (ruidosa en $Z$), sino una función del número de cortes donde la lesión es visible, asegurando que el volumen elipsoidal cubra físicamente esos cortes de 4mm.

*Nota: se probaron más versiones en las que se disminuia el rango de conectividad en cortes adyacentes (los centros debían estar más cerca de 0.8mm) para intentar solventar el caso I1009813, en el que se detecta la rCMB "desplazada" en los cortes adyacentes (probablemente serían vasos). Sin embargo, no se obtuvieron mejoras. Finalmente se decidió por poner un valor de 1 mm para ser estrictos pero sin pasarse.

In [ ]:
# nueva version: version 10
# volvemos al porcentaje de fondo pero lo hacemos mas estricto, usando un umbral axial del 70% en lugar del 80%
import os
import numpy as np
import nibabel as nib
import pandas as pd

# --- RUTAS ---
BASE_PATH = r"C:\Users\marta\Downloads\workdir"
IMAGE_DIR = os.path.join(BASE_PATH, "raw/positives")
MASK_DIR = os.path.join(BASE_PATH, "labels/positives_v10")
CSV_PATH = r"C:\Users\marta\Downloads\ADNI_MCH_Clean_Dataset.csv"

os.makedirs(MASK_DIR, exist_ok=True)

def analyze_lesion_v10(img_data, center_vox, pixdim):
    """Logica v10: Umbral estricto al 70% para evitar mascaras gigantes."""
    c_int = np.round(center_vox).astype(int)
    bg_roi = img_data[c_int[0]-10:c_int[0]+11, c_int[1]-10:c_int[1]+11, c_int[2]]
    bg_val = np.percentile(bg_roi, 75)
    
    # 1. Radios Axiales - Umbral mas estricto (70%)
    radii_mm = []
    for axis in [0, 1]:
        lengths = []
        for direction in [1, -1]:
            d = 1
            while d < 8:
                coord = list(c_int); coord[axis] += d * direction
                if not (0 <= coord[axis] < img_data.shape[axis]): break
                # Paramos mucho antes (al 70% del brillo del tejido sano)
                if img_data[tuple(coord)] >= bg_val * 0.70: break 
                d += 1
            lengths.append(d * pixdim[axis])
        radii_mm.append(np.mean(lengths))
    
    # 2. Conectividad Z (Margen 1.0mm y umbral 65%)
    z_slices = [c_int[2]]
    for z_dir in [1, -1]:
        nz = c_int[2] + z_dir
        if 0 <= nz < img_data.shape[2]:
            roi = img_data[c_int[0]-2:c_int[0]+3, c_int[1]-2:c_int[1]+3, nz]
            min_loc = np.unravel_index(np.argmin(roi), roi.shape)
            dist_radial = np.sqrt(((min_loc[0]-2)*pixdim[0])**2 + ((min_loc[1]-2)*pixdim[1])**2)
            
            if dist_radial < 1.0 and img_data[c_int[0]-2+min_loc[0], c_int[1]-2+min_loc[1], nz] < bg_val * 0.65:
                z_slices.append(nz)
    
    z_min, z_max = min(z_slices), max(z_slices)
    z_center_f = (z_min + z_max) / 2.0
    r_z_mm = ((z_max - z_min) * pixdim[2] + pixdim[2]*0.8) / 2.0
    
    return [radii_mm[0], radii_mm[1], r_z_mm], [center_vox[0], center_vox[1], z_center_f]

# --- PROCESAMIENTO MASIVO ---
df = pd.read_csv(CSV_PATH, sep=';')
df.columns = df.columns.str.strip()
df_coords = df[df['RASLOCATIONS'].notna()].copy()

all_files = [f for f in os.listdir(IMAGE_DIR) if f.endswith(".nii.gz")]

for i, filename in enumerate(all_files):
    img_id = filename.replace("I", "").replace(".nii.gz", "")
    rows = df_coords[df_coords['LONI_IMG_ID'].astype(str) == img_id]
    if rows.empty: continue

    img_nii = nib.load(os.path.join(IMAGE_DIR, filename))
    img_data = img_nii.get_fdata()
    pixdim, affine = img_nii.header.get_zooms()[:3], img_nii.affine
    final_mask = np.zeros(img_data.shape, dtype=np.uint8)

    for _, row in rows.iterrows():
        try:
            ras = np.array([float(x) for x in str(row['RASLOCATIONS']).split()[:3]])
            vox_f = (np.linalg.inv(affine) @ np.append(ras, 1))[:3]
            r_mm, c_f = analyze_lesion_v10(img_data, vox_f, pixdim)
            
            margin = [int(r/p) + 2 for r, p in zip(r_mm, pixdim)]
            X, Y, Z = np.ogrid[max(0,int(c_f[0]-margin[0])):min(img_data.shape[0],int(c_f[0]+margin[0])),
                               max(0,int(c_f[1]-margin[1])):min(img_data.shape[1],int(c_f[1]+margin[1])),
                               max(0,int(c_f[2]-margin[2])):min(img_data.shape[2],int(c_f[2]+margin[2]))]
            
            dist = ((X-c_f[0])*pixdim[0]/r_mm[0])**2 + ((Y-c_f[1])*pixdim[1]/r_mm[1])**2 + ((Z-c_f[2])*pixdim[2]/r_mm[2])**2
            final_mask[X, Y, Z] = np.maximum(final_mask[X, Y, Z], (dist <= 1.0))
        except Exception: continue

    nib.save(nib.Nifti1Image(final_mask, affine, img_nii.header), os.path.join(MASK_DIR, filename))
    if (i+1) % 20 == 0: print(f"Progreso: {i+1}/{len(all_files)} sujetos.")

print("Finalizado con umbral estricto (70%).")

Progreso: 20/184 sujetos.
Progreso: 40/184 sujetos.
Progreso: 60/184 sujetos.
Progreso: 80/184 sujetos.
Progreso: 100/184 sujetos.
Progreso: 120/184 sujetos.
Progreso: 140/184 sujetos.
Progreso: 160/184 sujetos.
Progreso: 180/184 sujetos.
Finalizado con umbral estricto (70%).


In [15]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
import matplotlib.pyplot as plt

# --- CONFIGURACIÓN DE RUTAS ---
BASE_PATH = r"C:\Users\marta\Downloads\workdir"
IMAGE_DIR = os.path.join(BASE_PATH, "raw/positives")
MASK_DIR = os.path.join(BASE_PATH, "labels/positives_v10")
CSV_PATH = r"C:\Users\marta\Downloads\ADNI_MCH_Clean_Dataset.csv"
OUTPUT_VIZ_DIR = os.path.join(BASE_PATH, "visualizations_v10")

os.makedirs(OUTPUT_VIZ_DIR, exist_ok=True)

def plot_validation_v10(img_data, mask_data, center_vox, filename):
    """Genera el estudio visual para comprobar el ajuste del umbral al 70%."""
    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    plt.suptitle(f"Validación v10 (Umbral 70%): {filename}")
    
    zc = int(np.round(center_vox[2]))
    for i, z_off in enumerate(range(-2, 3)):
        z = zc + z_off
        if 0 <= z < img_data.shape[2]:
            # Definir límites de recorte con seguridad para bordes de imagen
            x_s, x_e = int(center_vox[0])-30, int(center_vox[0])+30
            y_s, y_e = int(center_vox[1])-30, int(center_vox[1])+30
            
            crop = img_data[max(0,x_s):min(img_data.shape[0],x_e), 
                            max(0,y_s):min(img_data.shape[1],y_e), z]
            m_crop = mask_data[max(0,x_s):min(img_data.shape[0],x_e), 
                               max(0,y_s):min(img_data.shape[1],y_e), z]
            
            # Fila superior: Imagen original
            axes[0, i].imshow(crop.T, cmap='gray', origin='lower')
            axes[0, i].set_title(f"Z = {z}")
            
            # Fila inferior: Overlay de la máscara
            axes[1, i].imshow(crop.T, cmap='gray', origin='lower')
            axes[1, i].imshow(m_crop.T, cmap='Reds', alpha=0.4, origin='lower')
        
        for r in range(2): axes[r, i].axis('off')
            
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_VIZ_DIR, f"{filename}_v10_check.png"))
    plt.close()

# --- EJECUCIÓN ---
print("Cargando datos para generar visualizaciones v10...")
df = pd.read_csv(CSV_PATH, sep=';')
df.columns = df.columns.str.strip()
# Usamos solo los casos definitivos para centrar la visualización en la rCMB principal
df_coords = df[df['RASLOCATIONS'].notna()].copy()

mask_files = [f for f in os.listdir(MASK_DIR) if f.endswith(".nii.gz")]
print(f"Procesando {len(mask_files)} estudios visuales...")

for i, filename in enumerate(mask_files):
    img_id = filename.replace("I", "").replace(".nii.gz", "")
    rows = df_coords[df_coords['LONI_IMG_ID'].astype(str) == img_id]
    
    if rows.empty: continue

    # Carga de archivos (añadido por mí)
    img_nii = nib.load(os.path.join(IMAGE_DIR, filename))
    mask_nii = nib.load(os.path.join(MASK_DIR, filename))
    
    img_data = img_nii.get_fdata()
    mask_data = mask_nii.get_fdata()

    # Centramos la visualización en la primera rCMB anotada
    ras = np.array([float(x) for x in str(rows.iloc[0]['RASLOCATIONS']).split()[:3]])
    vox_f = (np.linalg.inv(img_nii.affine) @ np.append(ras, 1))[:3]
    
    plot_validation_v10(img_data, mask_data, vox_f, filename)

    if (i+1) % 20 == 0:
        print(f"Progreso: {i+1}/{len(mask_files)} imágenes generadas.")

print(f"Visualizaciones finalizadas. Revisa la carpeta: {OUTPUT_VIZ_DIR}")

Cargando datos para generar visualizaciones v10...
Procesando 184 estudios visuales...
Progreso: 20/184 imágenes generadas.
Progreso: 40/184 imágenes generadas.
Progreso: 60/184 imágenes generadas.
Progreso: 80/184 imágenes generadas.
Progreso: 100/184 imágenes generadas.
Progreso: 120/184 imágenes generadas.
Progreso: 140/184 imágenes generadas.
Progreso: 160/184 imágenes generadas.
Progreso: 180/184 imágenes generadas.
Visualizaciones finalizadas. Revisa la carpeta: C:\Users\marta\Downloads\workdir\visualizations_v10


In [5]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# --- CONFIGURACIÓN DE RUTAS ---
BASE_PATH = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/workdir_ADNI_subset"
IMAGE_DIR = os.path.join(BASE_PATH, "raw/positives")
MASK_DIR = os.path.join(BASE_PATH, "ADNI_dataset_pseudomasks/labels/positives_v10")
CSV_PATH = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/ADNI_MCH_Clean_Dataset.csv"
OUTPUT_3D_DIR = os.path.join(BASE_PATH, "ADNI_dataset_pseudomasks/visualizations_3D_v10")

os.makedirs(OUTPUT_3D_DIR, exist_ok=True)

def plot_3d_lesion(mask_data, center_vox, filename, index):
    """Crea una representación 3D de los vóxeles de la máscara."""
    # Definimos un cubo pequeño alrededor de la lesión para la visualización
    size = 10 
    cx, cy, cz = np.round(center_vox).astype(int)
    
    # Extraemos el sub-volumen (crop)
    voxels = mask_data[cx-size:cx+size, cy-size:cy+size, cz-size:cz+size]
    
    if not np.any(voxels):
        print(f"Aviso: No se encontraron vóxeles de máscara en el recorte para {filename}")
        return

    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    # Dibujamos los vóxeles
    ax.voxels(voxels, edgecolor='k', facecolors='red', alpha=0.7)
    
    ax.set_title(f"Reconstrucción 3D - Lesión {index}\nArchivo: {filename}")
    ax.set_xlabel('Eje X')
    ax.set_ylabel('Eje Y')
    ax.set_zlabel('Eje Z')
    
    plt.savefig(os.path.join(OUTPUT_3D_DIR, f"{filename}_lesion_{index}_3D.png"))
    plt.close()

# --- EJECUCIÓN ---
df = pd.read_csv(CSV_PATH, sep=',')
df.columns = df.columns.str.strip()
df_coords = df[df['RASLOCATIONS'].notna()].copy()

# Procesamos los primeros 10 archivos que tengan máscara
mask_files = [f for f in os.listdir(MASK_DIR) if f.endswith(".nii.gz")][:10]

print(f"Generando visualizaciones 3D para {len(mask_files)} archivos...")

for filename in mask_files:
    img_id = filename.replace("I", "").replace(".nii.gz", "")
    rows = df_coords[df_coords['LONI_IMG_ID'].astype(str) == img_id]
    
    if rows.empty: continue
    
    mask_nii = nib.load(os.path.join(MASK_DIR, filename))
    mask_data = mask_nii.get_fdata()
    affine = mask_nii.affine
    
    # Visualizamos cada lesión individual de ese archivo
    for idx, row in rows.iterrows():
        ras = np.array([float(x) for x in str(row['RASLOCATIONS']).split()[:3]])
        vox_f = (np.linalg.inv(affine) @ np.append(ras, 1))[:3]
        
        plot_3d_lesion(mask_data, vox_f, filename, idx)

print(f"Visualizaciones 3D finalizadas en: {OUTPUT_3D_DIR}")

Generando visualizaciones 3D para 10 archivos...
Aviso: No se encontraron vóxeles de máscara en el recorte para I1040756.nii.gz
Visualizaciones 3D finalizadas en: /media/PORT-DISK/Practicas/MicroBleeds_Generation/workdir_ADNI_subset/ADNI_dataset_pseudomasks/visualizations_3D_v10
